# STAC AI Workshop · NYC · 21 October 2026

## Section 4 · Actual LoRA adaptation: inspect, compare, qualify

**Participant workbook · Matthew Dixon, PhD, FRM**

STAC AI Workshop NYC · 21 October 2026

**Learning objectives**
1. Inspect actual SFT records and a saved LoRA configuration.
2. Compare base and adapter using identical prompts and evidence.
3. Recompute accuracy, per-class recall and balanced accuracy.
4. Separate evidence eligibility from forecast correctness.
5. State leakage and generalization limits before qualification.

Default mode replays **saved measured Qwen2.5-0.5B outputs**. No API, GPU or training runs. A/B/C are class-token outputs, not prose or calibrated probabilities. No measured separate RAG arm exists. Both compared models receive the same evidence. The instructor training notebook is separate.

## Open and run this notebook
[Open in Google Colab](https://colab.research.google.com/github/mfrdixon/STAC_AI_Bootcamp/blob/main/notebooks/04_actual_adaptation.ipynb) · [View on GitHub](https://github.com/mfrdixon/STAC_AI_Bootcamp/blob/main/notebooks/04_actual_adaptation.ipynb)

1. Run the **COLAB_BOOTSTRAP** cell first. It downloads public helpers and saved fixtures when using Colab; it does nothing in local JupyterLab.
2. Run the named code cells from top to bottom. Leave live, fresh-inference and GPU switches off initially.
3. References such as **C03:5** mean line 5 inside code cell C03. The guide quotes the exact code so you can find it without editor line numbers. Enable line numbers in your editor preferences if available.
4. Read **Answer checks** below to compare the default result. Edited inputs and live runs can differ; the stated conditions explain what should remain true.

Saved results, constructed examples, rule checks and fresh model calls are identified separately. Do not save API keys in cells or outputs.

In [ ]:
# COLAB_BOOTSTRAP · download the public workshop helpers and saved fixtures.
import sys
if "google.colab" in sys.modules:
    from pathlib import Path
    import os, subprocess
    workshop_root = Path("/content/STAC_AI_Bootcamp")
    if not (workshop_root / ".git").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/mfrdixon/STAC_AI_Bootcamp.git", str(workshop_root)], check=True)
    os.chdir(workshop_root / "notebooks")
    sys.path.insert(0, str(workshop_root))
    sys.path.insert(0, str(workshop_root / "notebooks"))
    print("Workshop helpers and saved fixtures ready. No model API call was made.")

<!-- WORKSHOP-CODE-GUIDE C01 -->
### Code cell C01 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C01:7** | `from adaptation_lab import load, verify, metrics, local_score` | Run optional fresh local inference only when explicitly enabled and model files exist. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C01 · code cell
from pathlib import Path
import sys, json
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "adaptation_lab.py").exists()), None)
if root is None: raise RuntimeError("Open this notebook inside the extracted workshop package.")
sys.path.insert(0, str(root))
from adaptation_lab import load, verify, metrics, local_score
import adaptation_display as view
data = load()
m = data["metadata"]
view.status(verify(data), m["execution"])


## 1 · Inspect what was trained
SFT supplies the objective; LoRA supplies the parameterization. Query/value adapters use rank 8, alpha 16. The original 494,032,768 parameters remain frozen. The adapter adds 540,672 trainable parameters, verified from saved tensor shapes.

<!-- WORKSHOP-CODE-GUIDE C02 -->
### Code cell C02 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C02:2** | `view.model(data)` | Display the recorded base-model and adapter configuration. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C02 · code cell
view.model(data)


## 2 · Inspect loss and checkpoint selection
These are recorded losses, not a new run. Epoch 2 was selected using earlier validation loss. Lower training loss does not establish improved held-out forecasting. Three non-finite gradient-norm log entries are retained as an audit limitation; saved evaluated logits are finite.

<!-- WORKSHOP-CODE-GUIDE C03 -->
### Code cell C03 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C03:2** | `view.losses(data)` | Display recorded losses; no fresh training is performed here. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C03 · code cell
view.losses(data)


## 3 · Choose a fixed held-out case
Change CASE_INDEX to inspect another recorded case. Predict negative, neutral or positive **before** revealing the observed outcome. The outcome is an evaluation label, not a fact the model knew.

<!-- WORKSHOP-CODE-GUIDE C04 -->
### Code cell C04 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C04:5** | `view.case_view(case)` | Display the matched saved base/adapted answers for the chosen held-out case. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C04 · code cell
CASE_INDEX = 0  # Choose 0–15; rerun this cell and the reveal cell below.
case = data["cases"][CASE_INDEX]
assert case["base"]["prompt_sha256"] == case["adapted"]["prompt_sha256"]
view.case_view(case)


## 4 · Reveal gold and inspect independent checks
Eligibility means the supplied records pass specified metadata/calculation checks. It does not prove that news is relevant or the forecast correct. For example, examine whether a news headline about other oil firms actually supports the target issuer.

<!-- WORKSHOP-CODE-GUIDE C05 -->
### Code cell C05 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C05:2** | `view.reveal(case)` | Reveal the reviewed target and checks after inspecting the saved responses. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C05 · code cell
view.reveal(case)


## 5 · Recompute aggregate results
All 16 cases remain in the evaluation; do not choose a winning subset. The test has only one neutral outcome. Correctly predicting it moves balanced accuracy substantially. Compare raw accuracy and class counts too.

<!-- WORKSHOP-CODE-GUIDE C06 -->
### Code cell C06 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C06:2** | `assert metrics(data,"base")["accuracy"] == 6/16` | Recompute a metric from the saved evaluation cases, not a new training run. |
| **C06:4** | `view.results(data, metrics)` | Display aggregate recorded performance on the same held-out cases. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C06 · code cell
assert metrics(data,"base")["accuracy"] == 6/16
assert metrics(data,"adapted")["accuracy"] == 3/16
view.results(data, metrics)


## 6 · Check temporal separation and qualification
Training outcomes precede April 1; validation outcomes precede June 5; July is the fixed evaluation block. The same four issuers appear across splits. Sources were downloaded later: these are not certified contemporaneous snapshots. A displayed historical publication date cannot prove historical content. No positive alpha was established.

<!-- WORKSHOP-CODE-GUIDE C07 -->
### Code cell C07 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C07:2** | `view.protocol(data)` | Display the recorded timing, split and qualification limits. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C07 · code cell
view.protocol(data)


## Optional · Fresh inference (explicitly off)
Requires approved local base and adapter files plus PyTorch, Transformers and PEFT. The function checks artifact hashes and rendered prompt hash before inference. It never downloads weights. CPU inference can be slow; the saved path is the workshop default. Changing input requires a separate fresh run and a new evaluation design; saved outputs do not update.

<!-- WORKSHOP-CODE-GUIDE C08 -->
### Code cell C08 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C08:2** | `RUN_FRESH=False` | This switch controls an optional live, model-download or training path. Leave it False for the default exercise. |
| **C08:6** | `view.fresh(local_score(case,BASE_DIR,ADAPTER_DIR,m["freeze"]["base_sha256"],m["freeze"]["adapter_sha256"]))` | Run optional fresh local inference only when explicitly enabled and model files exist. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C08 · code cell
RUN_FRESH=False
if RUN_FRESH:
 BASE_DIR=Path("/models/base") if Path("/models/base").exists() else root/"instructor-sandbox/models/base"
 ADAPTER_DIR=Path("/app/artifacts/alpha_v2/adapter") if Path("/app/artifacts/alpha_v2/adapter").exists() else root/"instructor-sandbox/artifacts/alpha_v2/adapter"
 view.fresh(local_score(case,BASE_DIR,ADAPTER_DIR,m["freeze"]["base_sha256"],m["freeze"]["adapter_sha256"]))
else:view.fresh()


## Learning check and exit ticket
- What was trained: a small adapter or a smaller replacement base model?
- Why is identical evidence important when comparing base and adapter?
- Why is higher balanced accuracy here insufficient evidence of a better model?
- Which evidence checks passed without proving relevance or forecasting skill?

**Debrief:** 540,672 adapter parameters were trained. The adapter changed behaviour, but July raw accuracy fell from 6/16 to 3/16. Balanced accuracy is sensitive to the single neutral case. Save a one-paragraph qualify/do-not-qualify decision with the specific evidence supporting it.

## Answer checks · default configuration

- A LoRA adapter was trained, not a smaller replacement base model. Rank and target modules are in the recorded metadata.
- Base and adapted answers use the **same prompt hash** for each held-out case. Changing evidence would confound the comparison.
- Recorded aggregate accuracy: base **6/16 (37.5%)**; adapted **3/16 (18.75%)**. Do not select only cases where the adapter improves.
- Check loss/checkpoint selection separately from held-out results: lower training loss does not certify workflow success.
- The reference target and eligibility checks are revealed after the case view. Case-specific outputs vary with CASE_INDEX; the reveal is the answer key for that selected case.
- `RUN_FRESH=False` uses saved artifacts. It does not run fresh inference or start a GPU server.

If your result differs, check which input, selected case, data version or optional switch changed. For a live model, inspect actual tool calls and returned evidence; do not assume the recorded result is guaranteed.